# B. Discovered Links Enrichment

This notebook enriches the raw discovered links dataset created by `A_source_link_discovery.ipynb`.

The goal is to create readable English features that can support later EDA, filtering, scoring, and candidate URL selection.

## 1. Imports

In [8]:
from pathlib import Path
import sys

import pandas as pd

sys.path.append("..")

from src.url_utils import get_last_url_path
from src.aws_translater import add_translated_column

# free alternative but subject to request rate limit:
# from src.google_translater import add_translated_column

## 2. Load Raw Discovered Links

In [9]:
RAW_DATA_PATH = Path("../data/raw/discovered_links_raw.csv")
INTERIM_DATA_DIR = Path("../data/interim")
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

ENRICHED_DATA_PATH = INTERIM_DATA_DIR / "discovered_links_enriched.csv"

df = pd.read_csv(RAW_DATA_PATH)

df.head()

,source_organisation,source_page,depth,link_text,href,full_url,url_section
0,kanker.nl,https://www.kanker.nl/,0,We zijn er voor je. Kanker.nl,/,https://www.kanker.nl/,homepage
1,kanker.nl,https://www.kanker.nl/,0,Blogs,/ervaringen-van-anderen/blogs,https://www.kanker.nl/ervaringen-van-anderen/b...,ervaringen-van-anderen
2,kanker.nl,https://www.kanker.nl/,0,In gesprek met elkaar,/ervaringen-van-anderen/gespreksgroepen,https://www.kanker.nl/ervaringen-van-anderen/g...,ervaringen-van-anderen
3,kanker.nl,https://www.kanker.nl/,0,Diagnose,/diagnose,https://www.kanker.nl/diagnose,diagnose
4,kanker.nl,https://www.kanker.nl/,0,Werk & kanker,/werk,https://www.kanker.nl/werk,werk


## 3. Create URL Slug Columns

`source_page_slug` describes the page where the link was found.

`full_url_slug` describes the destination URL that may be filtered, reviewed, or scraped later.

In [10]:
df["source_page_slug"] = df["source_page"].apply(get_last_url_path)
df["full_url_slug"] = df["full_url"].apply(get_last_url_path)

df[["source_page", "source_page_slug", "full_url", "full_url_slug"]].head()

,source_page,source_page_slug,full_url,full_url_slug
0,https://www.kanker.nl/,homepage,https://www.kanker.nl/,homepage
1,https://www.kanker.nl/,homepage,https://www.kanker.nl/ervaringen-van-anderen/b...,blogs
2,https://www.kanker.nl/,homepage,https://www.kanker.nl/ervaringen-van-anderen/g...,gespreksgroepen
3,https://www.kanker.nl/,homepage,https://www.kanker.nl/diagnose,diagnose
4,https://www.kanker.nl/,homepage,https://www.kanker.nl/werk,werk


## 4. Estimate Translation Cost

This estimates cost before translating. The estimate is based on unique values only, because the translation function translates unique values and maps them back to repeated rows.

In [11]:
# cost estimate for AWS Translate usage
# AWS Translate standard text translation is priced per source character.

AWS_TRANSLATE_PRICE_PER_MILLION_CHARS = 15.00

translated_source_columns = [
    "source_page_slug",
    "full_url_slug",
    "url_section",
    "link_text"
]

translation_cost_rows = []

for column in translated_source_columns:
    if column not in df.columns:
        continue

    unique_values = (
        df[column]
        .dropna()
        .astype(str)
        .drop_duplicates()
    )

    character_count = unique_values.str.len().sum()
    estimated_cost_usd = (
        character_count / 1_000_000
    ) * AWS_TRANSLATE_PRICE_PER_MILLION_CHARS

    translation_cost_rows.append({
        "translated_source_column": column,
        "unique_values_translated": len(unique_values),
        "source_character_count": character_count,
        "estimated_cost_usd": estimated_cost_usd,
    })

translation_cost_df = pd.DataFrame(translation_cost_rows)

total_characters = translation_cost_df["source_character_count"].sum()
total_estimated_cost_usd = translation_cost_df["estimated_cost_usd"].sum()

print("total translated source characters:", total_characters)
print("estimated AWS Translate cost USD:", round(total_estimated_cost_usd, 4))

translation_cost_df

total translated source characters: 247468
estimated AWS Translate cost USD: 3.712


,translated_source_column,unique_values_translated,source_character_count,estimated_cost_usd
0,source_page_slug,246,4193,0.062895
1,full_url_slug,3266,82525,1.237875
2,url_section,859,14607,0.219105
3,link_text,4168,146143,2.192145


## 5. Translate URL Section

In [12]:
df, url_section_translation_dict = add_translated_column(
    df,
    "url_section",
    "url_section_en",
)

df["url_section_en"] = df["url_section_en"].fillna(df["url_section"])

df[["url_section", "url_section_en"]].drop_duplicates().head(20)

translated 50/859 unique values
translated 100/859 unique values
translated 150/859 unique values
translated 200/859 unique values
translated 250/859 unique values
translated 300/859 unique values
translated 350/859 unique values
translated 400/859 unique values
translated 450/859 unique values
translated 500/859 unique values
translated 550/859 unique values
translated 600/859 unique values
translated 650/859 unique values
translated 700/859 unique values
translated 750/859 unique values
translated 800/859 unique values
translated 850/859 unique values


,url_section,url_section_en
0,homepage,homepage
1,ervaringen-van-anderen,others' experiences
3,diagnose,diagnosis
4,werk,work
5,jong,young
7,hulp-en-ondersteuning,help and support
11,agenda,planner
12,patientenorganisaties,patient organizations
13,ipso,ipso
14,kankersoorten,types of cancer


## 6. Translate Link Text

In [13]:
df, link_text_translation_dict = add_translated_column(
    df,
    "link_text",
    "link_text_en",
)

df["link_text_en"] = df["link_text_en"].fillna(df["link_text"])

df[["link_text", "link_text_en"]].drop_duplicates().head(20)

translated 50/4168 unique values
translated 100/4168 unique values
translated 150/4168 unique values
translated 200/4168 unique values
translated 250/4168 unique values
translated 300/4168 unique values
translated 350/4168 unique values
translated 400/4168 unique values
translated 450/4168 unique values
translated 500/4168 unique values
translated 550/4168 unique values
translated 600/4168 unique values
translated 650/4168 unique values
translated 700/4168 unique values
translated 750/4168 unique values
translated 800/4168 unique values
translated 850/4168 unique values
translated 900/4168 unique values
translated 950/4168 unique values
translated 1000/4168 unique values
translated 1050/4168 unique values
translated 1100/4168 unique values
translated 1150/4168 unique values
translated 1200/4168 unique values
translated 1250/4168 unique values
translated 1300/4168 unique values
translated 1350/4168 unique values
translated 1400/4168 unique values
translated 1450/4168 unique values
trans

,link_text,link_text_en
0,We zijn er voor je. Kanker.nl,We are here for you. Kanker.nl
1,Blogs,Blogs
2,In gesprek met elkaar,Talking to each other
3,Diagnose,Diagnosis
4,Werk & kanker,Work & cancer
5,Jong & kanker (16 t/m 39),Young & cancer (16 to 39)
6,Lotgenotenzoeker,Peer finder
7,Vraag het een professional,Ask a pro
8,Infolijn,Infoline
9,Vind hulp bij kanker,Find cancer help


## 7. Translate Source Page Slug

In [14]:
df, source_page_slug_translation_dict = add_translated_column(
    df,
    "source_page_slug",
    "source_page_slug_en",
)

df["source_page_slug_en"] = df["source_page_slug_en"].fillna(df["source_page_slug"])

df[["source_page_slug", "source_page_slug_en"]].drop_duplicates().head(20)

translated 50/246 unique values
translated 100/246 unique values
translated 150/246 unique values
translated 200/246 unique values


,source_page_slug,source_page_slug_en
0,homepage,homepage
63,blogs,blogs
132,gespreksgroepen,discussion groups
248,diagnose,diagnosis
311,werk,work
373,jong,young
446,lotgenotenzoeker,peer finder
504,stel-je-vraag-aan-een-professional,ask a professional your question
598,de-kankernl-infolijn,the-kankernl-info line
656,vind-hulp-bij-kanker,vind cancer help


## 8. Translate Full URL Slug

In [15]:
df, full_url_slug_translation_dict = add_translated_column(
    df,
    "full_url_slug",
    "full_url_slug_en",
)

df["full_url_slug_en"] = df["full_url_slug_en"].fillna(df["full_url_slug"])

df[["full_url_slug", "full_url_slug_en"]].drop_duplicates().head(20)

translated 50/3266 unique values
translated 100/3266 unique values
translated 150/3266 unique values
translated 200/3266 unique values
translated 250/3266 unique values
translated 300/3266 unique values
translated 350/3266 unique values
translated 400/3266 unique values
translated 450/3266 unique values
translated 500/3266 unique values
translated 550/3266 unique values
translated 600/3266 unique values
translated 650/3266 unique values
translated 700/3266 unique values
translated 750/3266 unique values
translated 800/3266 unique values
translated 850/3266 unique values
translated 900/3266 unique values
translated 950/3266 unique values
translated 1000/3266 unique values
translated 1050/3266 unique values
translated 1100/3266 unique values
translated 1150/3266 unique values
translated 1200/3266 unique values
translated 1250/3266 unique values
translated 1300/3266 unique values
translated 1350/3266 unique values
translated 1400/3266 unique values
translated 1450/3266 unique values
trans

,full_url_slug,full_url_slug_en
0,homepage,homepage
1,blogs,blogs
2,gespreksgroepen,discussion groups
3,diagnose,diagnosis
4,werk,work
5,jong,young
6,lotgenotenzoeker,peer finder
7,stel-je-vraag-aan-een-professional,ask a professional your question
8,de-kankernl-infolijn,the-kankernl-info line
9,vind-hulp-bij-kanker,vind cancer help


## 9. Reorder Columns

In [16]:
# enhanced discovered links columns order
column_order = [
    "source_organisation",
    "source_page",
    "source_page_slug",
    "source_page_slug_en",
    "depth",
    "link_text",
    "link_text_en",
    "href",
    "full_url",
    "full_url_slug",
    "full_url_slug_en",
    "url_section",
    "url_section_en",
]

# ensure all columsn included
existing_columns = [column for column in column_order if column in df.columns]
remaining_columns = [column for column in df.columns if column not in existing_columns]

df = df[existing_columns + remaining_columns]

# check df
df.head()

,source_organisation,source_page,source_page_slug,source_page_slug_en,depth,link_text,link_text_en,href,full_url,full_url_slug,full_url_slug_en,url_section,url_section_en
0,kanker.nl,https://www.kanker.nl/,homepage,homepage,0,We zijn er voor je. Kanker.nl,We are here for you. Kanker.nl,/,https://www.kanker.nl/,homepage,homepage,homepage,homepage
1,kanker.nl,https://www.kanker.nl/,homepage,homepage,0,Blogs,Blogs,/ervaringen-van-anderen/blogs,https://www.kanker.nl/ervaringen-van-anderen/b...,blogs,blogs,ervaringen-van-anderen,others' experiences
2,kanker.nl,https://www.kanker.nl/,homepage,homepage,0,In gesprek met elkaar,Talking to each other,/ervaringen-van-anderen/gespreksgroepen,https://www.kanker.nl/ervaringen-van-anderen/g...,gespreksgroepen,discussion groups,ervaringen-van-anderen,others' experiences
3,kanker.nl,https://www.kanker.nl/,homepage,homepage,0,Diagnose,Diagnosis,/diagnose,https://www.kanker.nl/diagnose,diagnose,diagnosis,diagnose,diagnosis
4,kanker.nl,https://www.kanker.nl/,homepage,homepage,0,Werk & kanker,Work & cancer,/werk,https://www.kanker.nl/werk,werk,work,werk,work


## 10. Simple Checks

In [17]:
print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Unique URLs:", df["full_url"].nunique())
print("Unique source pages:", df["source_page"].nunique())
print("Unique source_page_slug_en:", df["source_page_slug_en"].nunique())
print("Unique full_url_slug_en:", df["full_url_slug_en"].nunique())

Rows: 20856
Columns: 13
Unique URLs: 3851
Unique source pages: 257
Unique source_page_slug_en: 244
Unique full_url_slug_en: 3237


In [18]:
df["source_organisation"].value_counts()

source_organisation
iknl.nl                      5821
kanker.nl                    4566
nfk.nl                       4447
thuisarts.nl                 3626
zorgwijzer.nl                1451
zorginstituutnederland.nl     945
Name: count, dtype: int64

## 11. Save Enriched Discovered Links

In [19]:
df.to_csv(ENRICHED_DATA_PATH, index=False)

print(f"Saved {len(df):,} rows to {ENRICHED_DATA_PATH}")

Saved 20,856 rows to ../data/interim/discovered_links_enriched.csv
